In [3]:
from pathlib import Path
import numpy as np
import pandas as pd
import pyodbc
import sys
import os
import time
from datetime import date
import re
import warnings as wr
wr.filterwarnings('ignore')

In [4]:
seasons_as10 = {1:'Kharif', 2:'Phase-I', 3:'Phase-II', 4:'Phase-III', 5:'Summer'}
seasons_as20 = {1:'Kharif', 2:'E.Kharif', 3:'L.Kharif', 4:'Rabi', 5:'Summer'}

In [39]:
conn_str = (
    r'DRIVER={Microsoft Access Driver (*.mdb, *.accdb)};'
    f'DBQ={"./software/RC123kharif_TN.accdb"};'
)
conn = pyodbc.connect(conn_str)

#get data from the table - RC1
sql_query1 = 'SELECT * FROM "Copy Of RC1"'
planning = 260
b4_cod = 250
cols1 = ['RC', 'ST', 'YR', 'SESON', 'NOVP', 'NOVCOD', 'STAT', 'DIST', 'STRA',
       'VILL', 'EPC', 'NOVPC', 'NOVTRS', 'CADC', 'TLC', 'MAC', 'MUC', 'DDG',
       'SGC', 'GCC', 'LFOG', 'ROG', 'DDTRS', 'TRSSC', 'TRSSF', 'DCKC', 'HSSN',
       'GEOA', 'TCROPA', 'TNSSN', 'TGEOASN', 'RC61', 'RC62', 'RC63', 'REJ']
rc1 = (pd
    .read_sql(sql_query1, conn)[cols1]
    .apply(lambda x : pd.to_numeric(x, errors='coerce'))
    .assign(
        NOVP = planning,
        NOVCOD = b4_cod,
        REJ = lambda x : np.where(x['REJ']==1, 1, 0)
    )
)
conn.close()

In [41]:
rc1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 995 entries, 0 to 994
Data columns (total 35 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   RC       995 non-null    int64  
 1   ST       995 non-null    int64  
 2   YR       995 non-null    int64  
 3   SESON    995 non-null    int64  
 4   NOVP     995 non-null    int64  
 5   NOVCOD   995 non-null    int64  
 6   STAT     995 non-null    int64  
 7   DIST     995 non-null    int64  
 8   STRA     995 non-null    int64  
 9   VILL     995 non-null    int64  
 10  EPC      995 non-null    int64  
 11  NOVPC    995 non-null    int64  
 12  NOVTRS   995 non-null    int64  
 13  CADC     995 non-null    int64  
 14  TLC      995 non-null    int64  
 15  MAC      995 non-null    int64  
 16  MUC      970 non-null    float64
 17  DDG      995 non-null    int64  
 18  SGC      995 non-null    int64  
 19  GCC      995 non-null    int64  
 20  LFOG     995 non-null    int64  
 21  ROG      995 non

In [ ]:
def clean_rc123(year_int, season_cd, planning, b4_cod):
    #set season name
    season_nm = seasons[season_cd]

    #set database file path
    folder_path = Path('./software')
    for file_path in folder_path.iterdir():
        if file_path.is_file():
            if season_nm.lower() in file_path.name.lower():
                accdb_path = "./" + "/".join(str(file_path).split("\\"))
    
    #create connection with mdb database
    conn_str = (
        r'DRIVER={Microsoft Access Driver (*.mdb, *.accdb)};'
        f'DBQ={accdb_path};'
    )
    conn = pyodbc.connect(conn_str)
    
    #get data from the table - RC1
    sql_query1 = 'SELECT * FROM "Copy Of RC1"'
    cols1 = ['RC', 'ST', 'YR', 'SESON', 'NOVP', 'NOVCOD', 'STAT', 'DIST', 'STRA',
       'VILL', 'EPC', 'NOVPC', 'NOVTRS', 'CADC', 'TLC', 'MAC', 'MUC', 'DDG',
       'SGC', 'GCC', 'LFOG', 'ROG', 'DDTRS', 'TRSSC', 'TRSSF', 'DCKC', 'HSSN',
       'GEOA', 'TCROPA', 'TNSSN', 'TGEOASN', 'RC61', 'RC62', 'RC63', 'REJ']
    rc1 = (pd
        .read_sql(sql_query1, conn)[cols1]
        .apply(lambda x : pd.to_numeric(x, errors='coerce'))
        .assign(
            NOVP = planning,
            NOVCOD = b4_cod,
            REJ = lambda x : np.where(x['REJ']==1, 1, 0)
        )
    )



    
    
    #get data from the table - RC2
    sql_query2 = 'SELECT * FROM "Copy Of RC2"'
    rc2 = pd.read_sql(sql_query2, conn)
    
    #get data from the table - RC3
    sql_query3 = 'SELECT * FROM "Copy Of RC3"'
    rc3 = pd.read_sql(sql_query3, conn)
    
    conn.close()
    return rc1, rc2, rc3

In [ ]:
def clean_rc7(year_int):
    yr_int = year_int - 2001
    year = str(year_int)
    
    mdb_path = r"C:/ICS_Scheme/AS2.0/AS"+year+"/NSSAS"+year+".mdb"
    #create connection with mdb database
    conn_str = (
        r'DRIVER={Microsoft Access Driver (*.mdb, *.accdb)};'
        f'DBQ={mdb_path};'
    )
    conn = pyodbc.connect(conn_str)
    
    #get data from the table - Sch20RC7
    sql_query = "SELECT * FROM Sch20RC7"
    rc7 = (
        pd
        .read_sql(sql_query, conn)
        .astype({'EXPTID' : 'int64'})
    )
    conn.close()
    return rc7

In [3]:
def granularize(rc1, rc2, rc3, rc4):
    pass

In [4]:
def generate_annex(grc1, grc2, grc3, grc4):
    pass

In [ ]:
if __name__ == "__main__":
    #read & clean RCs
    year_int = 2024            #2024 for A.Y. 2024-25
    season_cd = [2,3]              #1-Kharif, 4-Rabi
    rc1_novp = {
        1 : [0,0],                     ##season_1 planning [cen, state]
        2 : [260, 260],                ##season_2 planning [cen, state]
        3 : [260, 260],                ##season_3 planning [cen, state]
        4 : [260, 260],                ##season_4 planning [cen, state]
        5 : [0,0]                      ##season_5 planning [cen, state]
    }
    rc1_novcod = {
        1 : [0,0],                     ##season_1 NOVCOD [cen, state]
        2 : [260, 260],                ##season_2 NOVCOD [cen, state]
        3 : [260, 260],                ##season_3 NOVCOD [cen, state]
        4 : [260, 260],                ##season_4 NOVCOD [cen, state]
        5 : [0,0]                      ##season_5 NOVCOD [cen, state]
    }

    rc1, rc2, rc3 = clean_rc123(year_int, season_cd, rc1_novp, rc1_novcod)
    rc7 = clean_rc7(year_int, season_cd)

    #Granularize RCs
    grc1, grc2, grc3, grc7 = granularize(crc1, crc2, crc3, crc7)

    #Generate Annexure Data
    generate_annex(grc1, grc2, grc3, grc7)